# Chapter 11: Design a News Feed System
**From:** *System Design Interview*

---

## TL;DR

- A news feed system has two core flows: **feed publishing** (write path) and **news feed retrieval** (read path).
- The **fanout service** is the critical component -- it pushes new posts to friends' news feed caches.
- **Fanout on write** (push) gives fast reads but suffers from the **hotkey problem** for celebrities; **fanout on read** (pull) saves resources for inactive users but has slow reads. A **hybrid approach** is optimal.
- The news feed cache stores only **post IDs** (not full objects) to save memory; full content is **hydrated** from separate user/post caches at read time.
- A **five-layer cache architecture** covers feed IDs, content, social graph, actions, and counters.

---
## Requirements

| Requirement | Value |
|---|---|
| Platforms | Mobile + Web |
| Features | Publish posts, view friends' feed |
| Feed ordering | Reverse chronological |
| Max friends | 5,000 per user |
| Traffic | 10 million DAU |
| Media | Images and videos supported |

---
## Back-of-the-Envelope Estimation

In [ ]:
# News Feed System Estimation
dau = 10_000_000  # 10M daily active users
avg_friends = 200  # average friends per user
posts_per_user_per_day = 2
avg_post_size_kb = 5  # text + metadata (media on CDN)

total_posts_per_day = dau * posts_per_user_per_day
# Each post fans out to avg_friends inboxes
fanout_writes_per_day = total_posts_per_day * avg_friends
fanout_qps = fanout_writes_per_day / (24 * 3600)

# Feed reads: assume each DAU checks feed 10 times/day
feed_reads_per_day = dau * 10
read_qps = feed_reads_per_day / (24 * 3600)

# Cache: store last 500 post IDs per user (8 bytes each)
cache_per_user_bytes = 500 * 8
total_cache_gb = (dau * cache_per_user_bytes) / (1024**3)

print(f"DAU:                      {dau:>14,}")
print(f"Total posts/day:          {total_posts_per_day:>14,}")
print(f"Fanout writes/day:        {fanout_writes_per_day:>14,}")
print(f"Fanout write QPS:         {fanout_qps:>14,.0f}")
print(f"Feed read QPS:            {read_qps:>14,.0f}")
print(f"Feed cache (IDs only):    {total_cache_gb:>14,.1f} GB")

---
## High-Level Design

### Feed Publishing Flow
```mermaid
graph LR
    User[User] -->|POST /v1/me/feed| LB[Load Balancer]
    LB --> Web[Web Servers]
    Web --> Post[Post Service]
    Web --> Fanout[Fanout Service]
    Web --> Notify[Notification Service]
    Post --> DB[(Post DB)]
    Post --> Cache[(Post Cache)]
    Fanout --> MQ[Message Queue]
    MQ --> Workers[Fanout Workers]
    Workers --> FeedCache[(News Feed Cache)]
```

### News Feed Retrieval Flow
```mermaid
graph LR
    User[User] -->|GET /v1/me/feed| LB[Load Balancer]
    LB --> Web[Web Servers]
    Web --> FeedSvc[News Feed Service]
    FeedSvc --> FeedCache[(Feed Cache - post IDs)]
    FeedSvc --> UserCache[(User Cache)]
    FeedSvc --> PostCache[(Post Cache)]
    FeedSvc --> CDN[CDN - media files]
    FeedSvc -->|hydrated feed| User
```

---
## Deep Dive: Fanout on Write vs Read

| Aspect | Fanout on Write (Push) | Fanout on Read (Pull) |
|---|---|---|
| When computed | At publish time | At read time |
| Read latency | Fast (pre-computed) | Slow (computed on demand) |
| Write cost | High (write to all friends) | Low (just persist the post) |
| Hotkey problem | Yes -- celebrities overwhelm system | No -- cost is on readers |
| Inactive users | Wasted writes | No wasted compute |
| Best for | Regular users with moderate friend count | Celebrities / high-follower accounts |

### Hybrid Approach
- **Push** for regular users (majority) -- fast feed reads
- **Pull** for celebrities (high follower count) -- avoid write amplification
- Use **consistent hashing** to distribute hotkey load evenly

## Deep Dive: Fanout Service Internals

```mermaid
sequenceDiagram
    participant U as User publishes post
    participant FS as Fanout Service
    participant GDB as Graph DB
    participant UC as User Cache
    participant MQ as Message Queue
    participant FW as Fanout Workers
    participant FC as Feed Cache

    U->>FS: New post (post_id)
    FS->>GDB: Get friend IDs
    GDB-->>FS: Friend list
    FS->>UC: Get friend settings
    UC-->>FS: Filter (muted, blocked)
    FS->>MQ: Send (post_id, friend_ids)
    MQ->>FW: Dequeue
    FW->>FC: Append post_id to each friend's feed
```

- Feed cache stores **<post_id, user_id>** pairs only -- not full objects
- Configurable limit on cached feed entries (users rarely scroll past recent posts)
- Cache miss rate stays low because most reads target recent content

## Deep Dive: Cache Architecture

The cache tier is organized into five layers:

| Layer | Stores | Access Pattern |
|---|---|---|
| **News Feed** | Post IDs for each user's feed | High read, append-on-write |
| **Content** | Full post data; hot cache for popular posts | Read-heavy |
| **Social Graph** | Friend/follower relationships | Read on fanout |
| **Action** | Likes, replies, shares per post | Read-heavy |
| **Counters** | Like count, reply count, follower count | Read-heavy, increment-on-write |

---
## Trade-offs

| Decision | Option A | Option B | Recommendation |
|---|---|---|---|
| Fanout strategy | Push (write) | Pull (read) | Hybrid -- push for most, pull for celebrities |
| Feed storage | Full post objects | Post IDs only | IDs only -- hydrate at read time to save memory |
| Database | SQL | NoSQL | Both -- SQL for user data, NoSQL for feed/posts |
| Feed ordering | Chronological | Ranked | Start chronological, add ranking later |
| Cache layers | Single cache | Multi-tier | Multi-tier -- different access patterns per data type |

---
## Key Takeaways

1. **The hybrid fanout model is the industry standard.** Push for regular users, pull for celebrities. This balances read latency against write amplification.

2. **Store IDs, not objects, in the feed cache.** Hydrate with full content at read time from separate caches. This dramatically reduces memory consumption.

3. **Graph databases excel at social relationships.** Friend lookups, friend-of-friend queries, and filtering (muted/blocked) are natural graph operations.

4. **Five-layer cache design.** Different data types have different access patterns -- separate cache tiers allow independent tuning and scaling.

5. **Message queues decouple publishing from fanout.** Workers process fanout asynchronously, preventing slow fanout from blocking the publish path.

---
## See Also

- [[fanout-on-write-vs-read]] -- Push vs pull models and the hybrid approach
- [[feed-publishing-flow]] -- The write path from post creation to friend notification
- [[newsfeed-retrieval]] -- The read path: ID lookup, hydration, and rendering
- [[graph-database-social]] -- Graph databases for social relationship queries
- [[cache-architecture]] -- Five-tier cache design for news feed systems